# 03. 후보 경로 만들기

            이 노트북의 목표는 **55개 지점 사이의 1,485개 경로 중에서 수업용 후보 network를 직접 만들어 보는 것**입니다.

            CSV 파일을 따로 열 필요는 없습니다. 아래 코드가 데이터를 읽고, 필요한 부분만 표와 그림으로 보여줍니다.


## 오늘 사용할 말

- graph(그래프): 점과 선으로 이루어진 연결 구조
- node(꼭짓점): 지도 위의 역 후보 지점
- edge(변): 두 지점을 연결하는 하나의 경로
- weight(가중치): 어떤 edge가 좋은지 나쁜지 판단하는 점수
- normalization(정규화): 서로 단위가 다른 값을 비교 가능한 점수로 바꾸는 일
- MST, minimum spanning tree(최소신장수형도): 모든 node를 연결하되 총 비용을 작게 만드는 기본 구조
- shortest path(최단경로): graph 안에서 두 node 사이를 가장 짧게 가는 경로
- stretch(우회율): 선택한 구조에서 얼마나 돌아가는지 나타내는 값
- t-spanner(t-스패너): 너무 많이 돌아가지 않도록 edge를 추가하는 방법


## 1. 준비하기

아래 셀은 수업에서 반복해서 쓸 도구를 불러옵니다.

처음 코딩을 해보는 사람은 이 셀의 모든 줄을 이해하지 않아도 됩니다. 지금은 “필요한 도구를 책상 위에 꺼내는 단계”라고 생각하면 됩니다.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for path in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (path / "analysis/student_helpers.py").is_file():
        PROJECT_ROOT = path
        break
    if (path / "student_helpers.py").is_file():
        PROJECT_ROOT = path
        break

if (PROJECT_ROOT / "analysis").is_dir():
    sys.path.insert(0, str(PROJECT_ROOT / "analysis"))
else:
    sys.path.insert(0, str(PROJECT_ROOT))

from student_helpers import *

OUT = output_dir(PROJECT_ROOT)
print("작업 폴더:", PROJECT_ROOT)
print("결과 저장 폴더:", OUT)


## 2. 데이터가 몇 개인지 확인하기

우리가 다루는 데이터는 다음 두 종류입니다.

- `stations`: 역 후보 지점 55개
- `edges`: 두 지점을 연결하는 모든 쌍 1,485개

`55 × 54 / 2 = 1,485`이므로, 모든 두 지점 쌍이 하나씩 들어 있어야 합니다.


In [ ]:
stations, edges = load_current_data(PROJECT_ROOT)
expected_pairs = len(stations) * (len(stations) - 1) // 2

print("station 개수:", len(stations))
print("edge 개수:", len(edges))
print("예상 edge 개수:", expected_pairs)

assert len(stations) == 55
assert len(edges) == expected_pairs == 1485

write_csv(OUT / "00_merged_edges.csv", edges)


## 3. edge 한 줄 살펴보기

edge 한 줄은 “A 지점에서 B 지점까지 가는 경로 하나”를 뜻합니다.

아래 표에서 볼 값은 네 가지입니다.

- W1: 거리
- W2: 평균 차로 수
- W3: 보호구역 proxy
- W4: 평균 경사


In [ ]:
sample_columns = [
    "pair_id", "from_station_name", "to_station_name",
    "distance_km", "w2_average_lanes",
    "w3_protection_proxy", "w4_average_absolute_grade_pct",
]
print_table(edges, sample_columns, limit=6)


## 4. 서로 다른 단위를 비교 가능한 점수로 바꾸기

거리는 km, 차로 수는 개수, 경사는 %입니다. 단위가 다르면 바로 더할 수 없습니다.

그래서 각 값을 비용 방향의 점수로 바꿉니다. 이것을 normalization이라고 합니다.

이번 기준은 다음과 같습니다.

- W1, W3: 값이 클수록 불리하므로 `값 / 최댓값`
- W2: 차로 수가 많을수록 유리하므로 `1 / 평균 차로 수`
- W4: 경사도 6%를 기준으로 `경사도 / 6`; 6% 초과는 penalty 값 `30`

즉 W4는 6%를 넘는 구간에 훨씬 큰 penalty를 줍니다.


In [ ]:
example = [100, 200, 400]
print("원래 값:", example)
print("최댓값으로 나누기:", [round(x / max(example), 3) for x in example])
lane_example = [1, 2, 4]
print("차로 수 역수:", [round(1 / x, 3) for x in lane_example])
slope_example = [2, 6, 7]
print("경사 penalty:", [round(slope_limit_penalty(x), 3) for x in slope_example])


## 5. balanced weight 정하기

이번 기본 세팅은 네 기준을 똑같이 봅니다.

```text
W1 거리             0.25
W2 차로 수          0.25
W3 보호구역 proxy   0.25
W4 예비 경사        0.25
```

차로 수는 많을수록 좋으므로, 코드에서는 “차로 수 역수 penalty”로 바꾸어 사용합니다.
너무 좁은 도로를 아예 후보에서 제외하고 싶으면 `MIN_AVERAGE_LANES` 값을 올리면 됩니다.


In [ ]:
WEIGHTS = {
    "distance": 0.25,
    "lane_capacity": 0.25,
    "protection_proxy": 0.25,
    "preliminary_slope": 0.25,
}
MAX_EDGE_DISTANCE_KM = 12.0
MIN_AVERAGE_LANES = 0.0
EXTRA_EDGE_COUNT = 30

print("weight 합계:", sum(WEIGHTS.values()))
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-12


## 6. 각 edge의 점수 계산하기

너무 먼 edge까지 후보로 넣으면 한 번에 너무 많은 선이 생깁니다. 여기서는 먼저 12km 이하 edge만 후보로 둡니다.

`scenario_cost`는 작을수록 이 기준에서 더 좋은 edge입니다.


In [ ]:
eligible = [
    edge for edge in edges
    if float(edge["distance_km"]) <= MAX_EDGE_DISTANCE_KM
]
scored = score_edges(eligible, WEIGHTS, min_average_lanes=MIN_AVERAGE_LANES)
write_csv(OUT / "03_edge_scores.csv", scored)

print("12km 이하 edge:", len(scored))
print_table(
    scored,
    ["pair_id", "from_station_name", "to_station_name", "distance_km", "w2_average_lanes", "slope_norm", "scenario_cost"],
    limit=8,
)


## 7. MST로 모든 지점을 한 번 연결하기

이제 점수가 낮은 edge를 중심으로 MST를 만듭니다. MST는 모든 node를 연결하되, loop를 만들지 않고 비용을 줄이는 기본 구조입니다.

하지만 MST만 쓰면 너무 빡빡한 구조가 되므로, 점수가 좋은 edge 30개를 추가로 더합니다.


In [ ]:
active_nodes, unresolved_nodes, candidate = build_candidate(stations, scored, EXTRA_EDGE_COUNT)
write_csv(OUT / "03_candidate_edges.csv", candidate)

print("연결된 node 수:", len(active_nodes))
print("연결되지 않은 node:", unresolved_nodes)
print("후보 edge 수:", len(candidate))
print_table(candidate, ["pair_id", "from_station_name", "to_station_name", "selection_role", "scenario_cost"], limit=10)


## 8. 결과 요약과 그림 만들기

마지막으로 이 후보 network가 어느 정도의 거리 구조를 갖는지 계산하고, 그림으로 저장합니다.

생각해볼 질문:

- MST edge와 extra edge는 어떤 역할이 다를까요?
- 비용이 낮은 edge만 고르면 실제 노선으로 바로 쓸 수 있을까요?


In [ ]:
active_edges = [edge for edge in scored if edge["from_station_id"] in active_nodes and edge["to_station_id"] in active_nodes]
metrics, _ = all_pair_metrics(active_nodes, active_edges, candidate)

summary = {
    "status": "PASS",
    "classification": "CURRENT_DATA_LECTURE_SCENARIO_NOT_FINAL_ROUTE",
    "station_count": len(stations),
    "active_station_count": len(active_nodes),
    "unresolved_station_ids": unresolved_nodes,
    "eligible_edge_count": len(scored),
    "candidate_edge_count": len(candidate),
    "weights": WEIGHTS,
    "maximum_edge_distance_km": MAX_EDGE_DISTANCE_KM,
    "minimum_average_lanes": MIN_AVERAGE_LANES,
"normalization_note": "W1,W3 divide by max; W2 uses inverse average lanes; W4 uses grade/6 up to 6% and assigns 30 when grade exceeds 6%.",
    "extra_edge_count": EXTRA_EDGE_COUNT,
    **metrics,
    "w3_note": "W3 is PRELIMINARY_POINT_BUFFER_PROXY, not official protected-zone geometry.",
    "w4_note": "W4 is PRELIMINARY_90M.",
}
write_json(OUT / "03_scenario_summary.json", summary)
plot_network(stations, candidate, OUT / "03_candidate_network.png", "03 candidate scenario")
summary
